# GGUF Export and Ollama Deployment

**Cipher Code Kraken - Final Export Stage**

Convert the fully-trained Cipher Code Kraken model to GGUF format for local inference via Ollama. Produces two quantization tiers:
- **Q4_K_M** (~18GB): Default quality tier. Good balance of quality and size.
- **Q5_K_M** (~22GB): Premium tier for users with 12GB+ VRAM. Better quality.

Also generates an Ollama Modelfile with the Cipher Code Kraken persona system prompt and Gemma 4 chat template.

**Input:** `cipher-final-merged` (from Stage 4: KTO)
**Output:** GGUF files + Ollama Modelfile

**Note:** Q3 quantization was explicitly dropped (5x perplexity degradation destroys persona coherence).

**Runtime:** Colab Pro+ A100 40GB | ~30-60 min

## Cell 1: Environment Setup
Install Unsloth which includes GGUF export capabilities via llama.cpp integration.

In [ ]:
!pip install unsloth

## Cell 2: Mount Drive and Load Final Model
Load the fully-trained model (after all 4 stages: SFT -> SimPO -> GRPO -> KTO).

**Important:** Load in full precision (not 4-bit) for GGUF conversion. The quantization happens during the GGUF export step itself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy final merged model from Drive
!cp -r /content/drive/MyDrive/cipher-models/cipher-final-merged ./
print("Final merged model copied from Drive")
!ls -la cipher-final-merged/ | head -10

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    "./cipher-final-merged",
    load_in_4bit=False,  # Load full precision for GGUF conversion
    max_seq_length=4096,
)
print(f"Model loaded in full precision. VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 3: Export Q4_K_M (Standard Quality)
Q4_K_M is the default quantization tier. Good balance of quality and size (~18GB for 31B model).

This is the recommended tier for most users (8GB+ VRAM GPUs).

In [ ]:
# Q4_K_M: Good balance of quality and size (~18GB for 31B model)
model.save_pretrained_gguf(
    "cipher-code-kraken-q4",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Q4_K_M export complete")
!ls -lh cipher-code-kraken-q4/

## Cell 4: Export Q5_K_M (Premium Quality)
Q5_K_M is the premium quantization tier. Better quality, larger size (~22GB for 31B model).

Recommended for users with 12GB+ VRAM who want maximum persona fidelity.

In [ ]:
# Q5_K_M: Better quality, larger size (~22GB for 31B model)
model.save_pretrained_gguf(
    "cipher-code-kraken-q5",
    tokenizer,
    quantization_method="q5_k_m",
)
print("Q5_K_M export complete")
!ls -lh cipher-code-kraken-q5/

## Cell 5: Create Ollama Modelfile
Generate the Ollama Modelfile with:
- Gemma 4 chat template format (`<start_of_turn>` / `<end_of_turn>`)
- Cipher Code Kraken system prompt defining the anti-slop persona
- Tuned inference parameters (temperature 0.7, top_p 0.9, 4096 context)

The system prompt explicitly instructs the model to:
- Specialize in Awwwards-worthy websites
- Use Three.js, GSAP, Lenis, WebGL shaders, advanced CSS
- NEVER generate generic template code or AI-slop patterns

In [ ]:
modelfile_content = '''FROM ./cipher-code-kraken-q4/unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<start_of_turn>system
{{ .System }}<end_of_turn>
{{ end }}{{ if .Prompt }}<start_of_turn>user
{{ .Prompt }}<end_of_turn>
{{ end }}<start_of_turn>model
{{ .Response }}<end_of_turn>"""

SYSTEM """You are Cipher Code Kraken, an expert creative web developer specializing in Awwwards-worthy websites. You write production-quality code using Three.js, GSAP (ScrollTrigger, SplitText, Flip), Lenis smooth scrolling, WebGL shaders, and advanced CSS techniques. You NEVER generate generic template code, AI-slop patterns, or utility-class-only designs. Every component you create is hand-crafted, interactive, and visually stunning.

Your personality is Maximum Kraken: ocean metaphors, tentacle references, deep-sea excitement about design. You teach through Socratic questioning, building understanding rather than just dumping code. When you see beautiful code, you celebrate it. When you see slop, you call it out.

You specialize in:
- Three.js: 3D scenes, particle systems, custom shaders, immersive experiences
- GSAP: ScrollTrigger, SplitText, Flip, timeline orchestration, spring physics
- Lenis: Buttery smooth scrolling, scroll-linked animations, parallax
- WebGL: Custom fragment/vertex shaders, noise-based effects, post-processing
- Advanced CSS: clip-path, mix-blend-mode, custom properties, container queries
- Vanilla JS: Custom cursors, magnetic buttons, intersection observers, canvas experiments

You NEVER produce: gradient heroes, div soup, cookie-cutter layouts, animate-bounce, generic card grids, or any pattern that looks like it came from a template."""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<start_of_turn>"
'''

with open("Modelfile", "w") as f:
    f.write(modelfile_content)
print("Modelfile created")
print("\nModelfile contents:")
print("=" * 60)
!cat Modelfile
print("=" * 60)
print("\nTo use with Ollama:")
print("  ollama create cipher-code-kraken -f Modelfile")
print("  ollama run cipher-code-kraken")

## Cell 6: Save All Exports to Drive
Copy GGUF files and Modelfile to Google Drive for download.

**Deployment steps (after downloading from Drive):**
1. Download GGUF file(s) and Modelfile to local machine
2. Place in the same directory
3. `ollama create cipher-code-kraken -f Modelfile`
4. `ollama run cipher-code-kraken`

For Q5_K_M premium tier, edit the Modelfile FROM line to point to the Q5 GGUF.

In [ ]:
# Save all exports to Drive
!cp -r cipher-code-kraken-q4 /content/drive/MyDrive/cipher-models/
!cp -r cipher-code-kraken-q5 /content/drive/MyDrive/cipher-models/
!cp Modelfile /content/drive/MyDrive/cipher-models/

# Also create a Q5 Modelfile variant
modelfile_q5 = modelfile_content.replace(
    'cipher-code-kraken-q4/unsloth.Q4_K_M.gguf',
    'cipher-code-kraken-q5/unsloth.Q5_K_M.gguf'
)
with open("Modelfile.q5", "w") as f:
    f.write(modelfile_q5)
!cp Modelfile.q5 /content/drive/MyDrive/cipher-models/

print("All exports saved to Drive")
print("\n" + "=" * 60)
print("CIPHER CODE KRAKEN TRAINING PIPELINE COMPLETE")
print("=" * 60)
print("\nPipeline stages completed:")
print("  1. SFT     -- Creative code instruction tuning")
print("  2. SimPO   -- Anti-slop preference optimization")
print("  3. GRPO    -- Reward-based creative quality reinforcement")
print("  4. KTO     -- Binary feedback alignment")
print("  5. GGUF    -- Quantized export for Ollama")
print("\nExports:")
print("  cipher-code-kraken-q4/ -- Q4_K_M (~18GB) [default]")
print("  cipher-code-kraken-q5/ -- Q5_K_M (~22GB) [premium]")
print("  Modelfile              -- Ollama config (Q4_K_M)")
print("  Modelfile.q5           -- Ollama config (Q5_K_M)")
print("\nDeployment:")
print("  1. Download GGUF + Modelfile from Drive")
print("  2. ollama create cipher-code-kraken -f Modelfile")
print("  3. ollama run cipher-code-kraken")
print("\nThe Kraken is ready to code.")